<a href="https://colab.research.google.com/github/slurya/experimental/blob/main/TrailHead_Binary_metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from openpyxl import load_workbook
import os


In [ ]:
import re
def parse_dasiy_answer(answer):
    result = re.search("^(true|false)", answer.lower())
    if not result:
        return answer
    else:
        # print(result)
        if result.group() == "true":
            return True
        elif result.group() == "false":
            return False
        else:
            return None

def parse_java_result(raw_result):
    if isinstance(raw_result, str):
        pattern = r"'knowledgeSummary'\s*:\s*'(True|False)'"
        result = re.search(pattern, raw_result.replace("\n", ""))

        if result:
            # print(result)
            if result.group(1).strip() == "False":
                return False
            elif result.group(1).strip() == "True":
                return True
            else:
                return None
        else:
            return raw_result
    else:
        return raw_result

def parse_llm_question(question):
    return question.replace("Please help answer with only true or false as the response: ", "").strip()

def proccess_groud_truth_question(question):
    return question.strip()


def get_non_bool_answer(joined_df):
    resp_columns = [dasiy_resp_col, java_resp_col]
    format_issue = {}
    can_not_answer = {}
    for resp_from in resp_columns:
        format_issue[resp_from] = []
        can_not_answer[resp_from] = []
        for agent_resp in joined_df[resp_from].to_list():
            if agent_resp not in (True, False):
                if "sorry" in agent_resp.lower():
                    can_not_answer[resp_from].append(agent_resp)
                else:
                    format_issue[resp_from].append(agent_resp)
    return format_issue, can_not_answer


def calculate_metric_agreement_yes_no(auto, human):
   human = np.asarray([str(label).strip().lower() for label in human])
   n_samples = len(human)
   for label in human:  # sanity check
       if label not in ['true', 'false']:
           print('WRONG LABEL!', label)

   auto = np.asarray([str(v).strip().lower()  for v in auto])
   n_wrong_label = 0
   for label in auto:
       if label not in ['true', 'false']:  # sanity check
           print('NONE LABEL!', label)
           n_wrong_label += 1
   print(f"{n_wrong_label} NONE labels")
   human = human[auto != None]
   auto = auto[auto != None]

   agreement = sum(human == auto) / n_samples
   precision = sum((human == 'true') & (auto == 'true')) / sum(auto == 'true')
   recall = sum((human == 'true') & (auto == 'true')) / sum(human == 'true')

   return agreement, precision, recall


dasiy_resp_col = "dasiy_response"
java_resp_col = "java_response"

In [ ]:
# groud_truth_path
ground_truth = "/content/drive/MyDrive/auto_evaluator/daisy_planner/dashboard/Data/TrailHead/filtered_ground_truth.csv"
# dasiy data path
dasiy_data = "/content/drive/MyDrive/auto_evaluator/daisy_planner/dashboard/Data/TrailHead/tf_daisy_500.xlsx"
# java data path
java_data = "/content/drive/MyDrive/auto_evaluator/daisy_planner/dashboard/Data/TrailHead/tf_500_queries_react.xlsx"

# change the sheet name accordingly
# ground_truth_df = pd.read_excel(open(ground_truth, "rb"), sheet_name="tf_sampled_500_seed_42")
ground_truth_df = pd.read_csv(ground_truth)
dasiy_result_df = pd.read_excel(open(dasiy_data, "rb"), sheet_name="tf_500_daisy")
java_df = pd.read_excel(open(java_data, "rb"), sheet_name="tf_trailhead_500")

# remove unused columns

dasiy_result_df = dasiy_result_df[["Generate: User Utterance", "Generate: Agent Response"]].rename(columns={"Generate: User Utterance": "question", "Generate: Agent Response": dasiy_resp_col})
java_df = java_df[["Generate: User Utterance", "Generate: Agent Response"]].rename(columns={"Generate: User Utterance": "question", "Generate: Agent Response": java_resp_col})
ground_truth_df = ground_truth_df[["question", "ground_truth"]]

In [ ]:
# parse the answer
dasiy_result_df["question"] = dasiy_result_df["question"].map(parse_llm_question)
dasiy_result_df[dasiy_resp_col] = dasiy_result_df[dasiy_resp_col].map(parse_dasiy_answer)

java_df["question"] = java_df['question'].map(parse_llm_question)
java_df[java_resp_col] = java_df[java_resp_col].map(parse_java_result)

ground_truth_df["question"] = ground_truth_df['question'].map(proccess_groud_truth_question)


In [ ]:
joined_df = dasiy_result_df.merge(java_df, on="question", how="inner")
joined_df = joined_df.merge(ground_truth_df, on="question", how="inner")

In [ ]:
# save the joined df
joined_df.to_csv(f"/content/drive/MyDrive/auto_evaluator/daisy_planner/dashboard/Intermediate_Results/TrailHead/Binary_joined_{datetime.today()}.csv")

In [ ]:
format_issue, can_not_answer = get_non_bool_answer(joined_df)

In [ ]:
dasiy_filter_df = joined_df[joined_df[dasiy_resp_col].isin([True, False])]
dasiy_agreement, dasiy_precision, dasiy_recall = calculate_metric_agreement_yes_no(dasiy_filter_df[dasiy_resp_col].to_list(), dasiy_filter_df["ground_truth"].to_list())

java_filter_df = joined_df[joined_df[java_resp_col].isin([True, False])]
java_agreement, java_precision, java_recall = calculate_metric_agreement_yes_no(java_filter_df[java_resp_col].to_list(), java_filter_df["ground_truth"].to_list())

0 NONE labels
0 NONE labels


In [ ]:
#
today = datetime.today().strftime("%Y-%m-%d")
result_data = []
result_data.append({
    "Dataset": "TrailHead T/F",
    "Eval Date": today,
    "Data size": len(joined_df),
    "Planner version": "dasiy",
    "Correct response": len(dasiy_filter_df),
    "Bad output format": len(format_issue[dasiy_resp_col]),
    "can't answer": len(can_not_answer[dasiy_resp_col]),
    "Coverage": f"{1 - (len(format_issue[dasiy_resp_col]) + len(can_not_answer[dasiy_resp_col])) / len(joined_df):.2%}" ,
    "Accuracy": f"{dasiy_agreement:.2%}" ,
    "Precision": f"{dasiy_precision:.2%}" ,
    "Recall":  f"{dasiy_recall:.2%}" ,
    "Comment": str({"format issue": format_issue[dasiy_resp_col], "can't answer": can_not_answer[dasiy_resp_col]})
})

result_data.append({
    "Dataset": "TrailHead T/F",
    "Eval Date": today,
    "Data size": len(joined_df),
    "Planner version": "java",
    "Correct response": len(java_filter_df),
    "Bad output format": len(format_issue[java_resp_col]),
    "can't answer": len(can_not_answer[java_resp_col]),
    "Coverage": f"{1 - (len(format_issue[java_resp_col]) + len(can_not_answer[java_resp_col])) / len(joined_df):.2%}" ,
    "Accuracy":  f"{java_agreement:.2%}" ,
    "Precision": f"{java_precision:.2%}" ,
    "Recall": f"{java_recall:.2%}" ,
    "Comment": str({"format issue": format_issue[java_resp_col], "can't answer": can_not_answer[java_resp_col]})
})
result_df = pd.DataFrame(result_data)

In [ ]:
result_df

,Dataset,Eval Date,Data size,Planner version,Correct response,Bad output format,can't answer,Coverage,Accuracy,Precision,Recall,Comment
0,TrailHead T/F,2025-03-27,376,dasiy,350,23,3,93.09%,84.86%,85.88%,92.80%,{'format issue': ['Your strategy for dealing w...
1,TrailHead T/F,2025-03-27,376,java,307,11,58,81.65%,92.18%,93.61%,95.35%,{'format issue': ['False. The specific tests y...


In [ ]:
def append_to_excel(df, file_path, sheet_name='Sheet1'):
    file_exists = os.path.exists(file_path)
    mode = 'a' if file_exists else 'w'
    if_sheet_exists = 'overlay' if file_exists else None
    header = False if file_exists else True

    with pd.ExcelWriter(
        file_path,
        engine='openpyxl',
        mode=mode,
        if_sheet_exists=if_sheet_exists
    ) as writer:
        if file_exists:
            # writer.book = load_workbook(file_path)
            if sheet_name in writer.book.sheetnames:
                startrow = writer.book[sheet_name].max_row
            else:
                startrow = 0
                header = True
        else:
            startrow = 0

        df.to_excel(
            writer,
            sheet_name=sheet_name,
            startrow=startrow,
            index=False,
            header=header
        )


In [ ]:
file_path = "/content/drive/MyDrive/auto_evaluator/daisy_planner/dashboard/Output/TrailHead/binary_dashboard.xlsx"

append_to_excel(pd.DataFrame(result_data), file_path)